In [2]:
import torch
import torchvision
import torchvision.transforms.v2
from torch.utils.data import DataLoader

transforms = torchvision.transforms.v2.Compose([
    torchvision.transforms.v2.Resize(224),
    torchvision.transforms.v2.Grayscale(num_output_channels=3),
    torchvision.transforms.v2.ToImage(),
    torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
    torchvision.transforms.v2.Normalize((0.1307,), (0.3081,)),
])
train_ds = torchvision.datasets.MNIST(
    "mnist", train=True, download=True, transform=transforms
)
test_ds = torchvision.datasets.MNIST(
    "mnist", train=False, download=True, transform=transforms
)

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=128, shuffle=False)

In [1]:
import torchvision.models

torchvision.models.resnet18()

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
resnet = torchvision.models.resnet18(
    weights=torchvision.models.ResNet18_Weights.DEFAULT
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/abramov-alex/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:05<00:00, 7.81MB/s]


In [7]:
x = next(iter(train_dl))[0]
print(x.shape)

x = resnet.conv1(x)
print(x.shape)
x = resnet.bn1(x)
print(x.shape)
x = resnet.relu(x)
print(x.shape)
x = resnet.maxpool(x)
print(x.shape)

x = resnet.layer1(x)
print('!!!!', x.shape)
x = resnet.layer2(x)
print('!!!!', x.shape)
x = resnet.layer3(x)
print('!!!!', x.shape)
x = resnet.layer4(x)
print('!!!!', x.shape)

x = resnet.avgpool(x)
print(x.shape)
x = torch.flatten(x, 1)
print(x.shape)
x = resnet.fc(x)
print(x.shape)

torch.Size([128, 3, 224, 224])
torch.Size([128, 64, 112, 112])
torch.Size([128, 64, 112, 112])
torch.Size([128, 64, 112, 112])
torch.Size([128, 64, 56, 56])
!!!! torch.Size([128, 64, 56, 56])
!!!! torch.Size([128, 128, 28, 28])
!!!! torch.Size([128, 256, 14, 14])
!!!! torch.Size([128, 512, 7, 7])
torch.Size([128, 512, 1, 1])
torch.Size([128, 512])
torch.Size([128, 1000])


In [ ]:
import torch

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
resnet.eval()
resnet = resnet.to(device)
print(f"Using device: {device}")

# Hook all 8 residual blocks: 56x56, 28x28, 14x14, 7x7 (two blocks each stage)
HOOK_TARGETS = [
    ("layer1.0", resnet.layer1[0]),
    ("layer1.1", resnet.layer1[1]),
    ("layer2.0", resnet.layer2[0]),
    ("layer2.1", resnet.layer2[1]),
    ("layer3.0", resnet.layer3[0]),
    ("layer3.1", resnet.layer3[1]),
    ("layer4.0", resnet.layer4[0]),
    ("layer4.1", resnet.layer4[1]),
]
LAYER_NAMES = [name for name, _ in HOOK_TARGETS]
print(f"Hooking {len(HOOK_TARGETS)} layers:", LAYER_NAMES)

In [ ]:
import tqdm

N_SUBSAMPLE = 2000

layer_outputs = {name: [] for name in LAYER_NAMES}

def make_hook(name):
    def hook(module, inp, out):
        layer_outputs[name].append(out.detach().cpu())
    return hook

hooks = [module.register_forward_hook(make_hook(name)) for name, module in HOOK_TARGETS]

count = 0
with torch.no_grad():
    for X, y in tqdm.tqdm(test_dl, desc="Extracting features"):
        if count >= N_SUBSAMPLE:
            break
        resnet(X.to(device))
        count += X.shape[0]

for h in hooks:
    h.remove()

for name in LAYER_NAMES:
    layer_outputs[name] = torch.cat(layer_outputs[name])[:N_SUBSAMPLE]
    print(f"{name}: {layer_outputs[name].shape}")

In [ ]:
import numpy as np
import sys
sys.path.insert(0, '.')
from utils.cubical_DL import feature_map_to_pd

layer_pds = {}
for name in LAYER_NAMES:
    feat = layer_outputs[name]
    pds = [feature_map_to_pd(feat[i]) for i in tqdm.trange(len(feat), desc=f"Cubical PD [{name}]")]
    layer_pds[name] = pds
    n_pts = np.mean([len(pd) for pd in pds])
    print(f"  {name}: avg {n_pts:.1f} PD points/image, spatial grid {tuple(feat.shape[2:])}")

In [ ]:
from utils.cubical_DL import pds_to_knn_graph

KNN = 4

layer_knn_graphs = {}
for name in LAYER_NAMES:
    G = pds_to_knn_graph(layer_pds[name], k=KNN, method='pi')
    layer_knn_graphs[name] = G
    print(f"{name}: k-NN graph {G.shape}, nnz={G.nnz}")

In [ ]:
import gc
torch.mps.empty_cache() if device.type == 'mps' else None
gc.collect()

from utils.topological_analyzer import TopologicalAnalyzer

knn_graph_list = [layer_knn_graphs[name] for name in LAYER_NAMES]
params = {"knn": KNN, "dim": 3}

bars = TopologicalAnalyzer.compute_zigzag_barcodes_from_knn_graphs(
    knn_graphs=knn_graph_list,
    params=params,
)
print("Zigzag done. Diagram sizes by hom_dim:", [len(dgm) for dgm in bars['raw_diagrams']])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
from utils.post_process import POSTPROCESS

hex6 = ['#648FFF', '#785EF0', '#DC267F', '#FE6100', '#FFB000']
colors6 = [mcolors.to_rgb(i) for i in hex6]
cmap0 = mcolors.LinearSegmentedColormap.from_list("", list(zip([0, 1], ['white', colors6[2]])))
cb_palette = [
    '#648FFF', '#785EF0', '#DC267F', '#FE6100', '#FFB000',
    '#3CAB20', '#6B750C', '#A6761D', '#D8A21E', '#F0E442',
    '#1F77B4', '#FF7F0E', '#2CA02C', '#D62728', '#9467BD',
    '#8C564B', '#E377C2', '#7F7F7F', '#BCBD22', '#17BECF',
]

num_layers = len(LAYER_NAMES)
print(f"num_layers = {num_layers}, layer names: {LAYER_NAMES}")

In [ ]:
# --- Persistence image in (birth, persistence) coordinates (H1 barcodes) ---
pp = POSTPROCESS(pds=bars['raw_diagrams'], num_layers=num_layers * 2, start_ind=1, zigzag=True, debug=False)
pis = pp.find_eff_pis()[1]
bettis = pp.find_betti_layers()[1]

warnings.filterwarnings('ignore')

pis_pers = np.zeros((len(pis), num_layers, num_layers))
for i in range(num_layers):
    for j in range(num_layers):
        if i - j >= 0:
            pis_pers[:, i - j, j] = pis[i, j]

plt.figure(figsize=(7, 6))
plt.imshow(np.log10(pis_pers[1] + 1e-10), cmap=cmap0, origin='lower')
cb = plt.colorbar()
cb.set_label('Log10 number of 1-cycles', fontsize=13)
cb.ax.tick_params(labelsize=13)
plt.xticks(ticks=range(num_layers), labels=LAYER_NAMES, rotation=45, ha='right', fontsize=11)
plt.yticks(fontsize=13)
plt.xlabel('Birth Layer $(\ell_{\\rm birth})$', fontsize=13)
plt.ylabel('Persistence $(\ell_{\\rm death} - \ell_{\\rm birth})$', fontsize=13)
plt.title('ResNet-18: zigzag 1-cycles (cubical bridge)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def inter_layer_persistence(raw_diagrams, num_layers, p=1):
    pp = POSTPROCESS(pds=raw_diagrams, num_layers=num_layers * 2, start_ind=1, zigzag=True, debug=False)
    psim = pp.find_ph_sim()[p]
    bettis = pp.find_betti_layers()[p]
    temp = np.zeros((num_layers, num_layers))
    for i in range(num_layers):
        if bettis[i] == 0:
            temp[i] = 0
        else:
            temp[i] = psim[i] / bettis[i]
    return temp


def weighted_inter_layer_persistence(layer_pers, n_layers, p=1):
    weights = np.zeros((n_layers, n_layers))
    for i in range(n_layers):
        for j in range(n_layers):
            weights[i][j] = 1e-10 if (p < 0 and i == j) else np.abs(i - j) ** p
    return np.sum(layer_pers * weights, axis=1) / np.sum(weights, axis=1)


plt.figure(figsize=(7, 6))
ps = [-1., 0., 0.5, 1, 2]
for p in ps:
    pers = inter_layer_persistence(bars['raw_diagrams'], num_layers)
    w_pers = weighted_inter_layer_persistence(pers, num_layers, p)
    std = np.std(w_pers)
    x = np.arange(num_layers)
    plt.plot(x, w_pers, color=cb_palette[ps.index(p)], label=f'$\\alpha={p:.2f}$')
    plt.fill_between(x, w_pers - std, w_pers + std, color=cb_palette[ps.index(p)], alpha=0.2)

plt.xticks(ticks=range(num_layers), labels=LAYER_NAMES, rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=18)
plt.xlabel('Layer', fontsize=17)
plt.ylabel('Inter-Layer Persistence', fontsize=17)
plt.legend(fontsize=14)
plt.title('ResNet-18: inter-layer persistence', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def effective_pi(raw_diagrams, num_layers, p=1):
    pp = POSTPROCESS(pds=raw_diagrams, num_layers=num_layers * 2, start_ind=1, zigzag=True, debug=False)
    pis = pp.find_eff_pis()[p]
    pis_death = np.zeros((num_layers, num_layers))
    for i in range(num_layers):
        for j in range(num_layers):
            pis_death[i, j] = pis[i, j]
    return pis_death


def pi_hist(raw_diagrams, num_layers, p=2, axis=1):
    weights = np.zeros((num_layers, num_layers))
    for i in range(num_layers):
        for j in range(num_layers):
            weights[i][j] = 1e-10 if (p < 0 and i == j) else np.abs(i - j) ** p
    pis = effective_pi(raw_diagrams, num_layers)
    pis_h = np.sum(pis * weights, axis=axis) / np.sum(weights, axis=axis)
    pis_h /= np.sum(pis_h) if np.sum(pis_h) > 0 else 1.0
    return pis_h


plt.figure(figsize=(7, 6))
ps = [-1., 0., 0.5, 1, 2]
for p in ps:
    ph = pi_hist(bars['raw_diagrams'], num_layers, p)
    std = np.std(ph)
    x = np.arange(num_layers)
    plt.plot(x, ph, color=cb_palette[ps.index(p)], label=f'$\\alpha={p:.1f}$')
    plt.fill_between(x, ph - std, ph + std, color=cb_palette[ps.index(p)], alpha=0.2)

plt.axhline(y=1 / num_layers, label='Uniform distribution', color='black', linestyle='--', lw=1)
plt.xticks(ticks=range(num_layers), labels=LAYER_NAMES, rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=18)
plt.xlabel('Layer', fontsize=17)
plt.ylabel('Births Relative Frequency', fontsize=17)
plt.legend(fontsize=14)
plt.title('ResNet-18: birth frequency', fontsize=13)
plt.tight_layout()
plt.show()